In [ ]:
from theia.simulation.logging import LogLoader


loader = LogLoader("output.json")

In [ ]:
import numpy as np


history = np.array([[4289013.05059586, 0.0, 623090.7325218604, 0.0, 4665068.725742616, 0.0], [4289138.289496881, 0.0, 623321.0487881298, 0.0, 4664889.124387505, 0.0], [4289166.999911805, 0.0, 623586.8012248795, 0.0, 4664819.409466581, 0.0], [4289197.007857473, 0.0, 623849.2908026148, 0.0, 4664748.985410989, 0.0], [4289228.335298004, 0.0, 624108.2444447224, 0.0, 4664677.875040875, 0.0], [4289260.998068351, 0.0, 624363.3824140638, 0.0, 4664606.108054733, 0.0], [4289295.005101269, 0.0, 624614.4222425737, 0.0, 4664533.721134467, 0.0], [4289330.357793152, 0.0, 624861.0834231083, 0.0, 4664460.757796866, 0.0], [4289367.049570515, 0.0, 625103.0927688545, 0.0, 4664387.267948065, 0.0], [4289405.065716196, 0.0, 625340.1902736619, 0.0, 4664313.307111572, 0.0], [4289444.383504791, 0.0, 625572.1352355225, 0.0, 4664238.935320918, 0.0], [4289484.972679947, 0.0, 625798.7123452934, 0.0, 4664164.215693614, 0.0], [4289526.796282959, 0.0, 626019.7374046386, 0.0, 4664089.2127311425, 0.0], [4289569.811814847, 0.0, 626235.0623301909, 0.0, 4664013.990416765, 0.0], [4289613.9726861315, 0.0, 626444.579130545, 0.0, 4663938.610204627, 0.0], [4289659.229883811, 0.0, 626648.2226079779, 0.0, 4663863.129006713, 0.0], [4289705.533767138, 0.0, 626845.9716308285, 0.0, 4663787.5972861275, 0.0], [4289752.835895227, 0.0, 627037.8489324838, 0.0, 4663712.0573557075, 0.0], [4289801.090791157, 0.0, 627223.9195035716, 0.0, 4663636.541961771, 0.0], [4289850.257557813, 0.0, 627404.2877399178, 0.0, 4663561.073207381, 0.0], [4289900.301277794, 0.0, 627579.0935780959, 0.0, 4663485.661842276, 0.0], [4289951.194149308, 0.0, 627748.5078858897, 0.0, 4663410.30692233, 0.0], [4290002.916327894, 0.0, 627912.727374693, 0.0, 4663334.99582445, 0.0], [4290055.456455688, 0.0, 628071.9692665297, 0.0, 4663259.704597088, 0.0], [4290108.811860556, 0.0, 628226.4658819371, 0.0, 4663184.3986365255, 0.0], [4290162.988390004, 0.0, 628376.4592147119, 0.0, 4663109.033711681, 0.0], [4290217.999796334, 0.0, 628522.1954109562, 0.0, 4663033.5574298715, 0.0], [4290273.866482401, 0.0, 628663.9188305344, 0.0, 4662957.911375798, 0.0], [4287458.1385822315, 0.0, 630386.3948468856, 0.0, 4665285.08442861, 0.0], [4287371.584785, 0.0, 630436.7869661902, 0.0, 4665355.074001577, 0.0], [4287290.143469071, 0.0, 630484.0579795926, 0.0, 4665420.9451423185, 0.0], [4287213.883102129, 0.0, 630528.190644113, 0.0, 4665482.640038135, 0.0], [4287142.867366835, 0.0, 630569.1706744363, 0.0, 4665540.104727025, 0.0], [4287077.1568098385, 0.0, 630606.985532289, 0.0, 4665593.287791364, 0.0], [4287016.810656482, 0.0, 630641.6231252365, 0.0, 4665642.138916977, 0.0]])[:, [0, 2, 4]]

In [ ]:
from matplotlib import pyplot as plt


plt.plot(np.arange(history.shape[0]), history[:, 0])

In [ ]:
import itertools


pcl_detections = sorted(loader.blue_pcl_detections, key=lambda d: d.time)
for time, detections in itertools.groupby(pcl_detections, key=lambda d: d.time):
    detections = list(detections)
    if len(detections) >= 3:
        break

In [ ]:
assert all([detection.target == detections[0].target for detection in detections])
target = detections[0].target

In [ ]:
loader.blue_pcl_sensors

In [ ]:
import numpy as np

from theia.coordinates import CoordinateTransformations


p_true = np.array(CoordinateTransformations.geodetic_to_cartesian(*target.point.as_tuple()))
p_calc = np.array((4289013.05059586, 623090.7325218604, 4665068.725742616))
assert np.isclose(p_true, p_calc).all()

In [ ]:
loader.s

In [ ]:
from theia.mapping import RadarMap


RadarMap(
    sensors={f"ID {sensor.id}": sensor for sensor in loader.blue_pcl_sensors},
    targets={"target": target},
).to_map()

In [ ]:
from theia.detection.pcl import PclDetector
from theia.test_data import load_pcl_example

sensors, trajcetories, grid = load_pcl_example()
trajectory = trajcetories[0]
RCS = trajectory.cross_section_model.rcs

In [ ]:
grid.model_dump_json()

In [ ]:
from typing import Any, Literal

import pydantic
import shapely

import theia
from theia.detection.pcl import pcl_track_init_update_masks
from theia.grids import LatLonHeightGrid
from theia.types import Sensor
from theia.util import mask_to_polygon


class GeoJSONPolygon(pydantic.BaseModel):
    type: Literal["Polygon"] = "Polygon"
    coordinates: list[list[list[float]]]

    @classmethod
    def from_shapely(cls, polygon: shapely.Polygon) -> "GeoJSONPolygon":
        geojson = shapely.geometry.mapping(polygon)
        return cls(
            coordinates=[list(map(list, ring)) for ring in geojson["coordinates"]]
        )


class GeoJSONMultiPolygon(pydantic.BaseModel):
    type: Literal["MultiPolygon"] = "MultiPolygon"
    # One extra nesting level: [polygon][ring][point][coordinate]
    coordinates: list[list[list[list[float]]]]

    @classmethod
    def from_shapely(cls, multi: shapely.MultiPolygon) -> "GeoJSONMultiPolygon":
        geojson = shapely.geometry.mapping(multi)
        return cls(
            coordinates=[
                [list(map(list, ring)) for ring in polygon]
                for polygon in geojson["coordinates"]
            ]
        )


GeoJSONGeometry = GeoJSONPolygon | GeoJSONMultiPolygon


class GeoJSONFeature(pydantic.BaseModel):
    type: str = "Feature"
    geometry: GeoJSONGeometry = pydantic.Field(discriminator="type")
    properties: dict[str, Any] = {}

    @classmethod
    def from_shapely(
        cls,
        shape: shapely.Polygon | shapely.MultiPolygon,
        properties: dict[str, Any] = {},
    ) -> "GeoJSONFeature":
        if isinstance(shape, shapely.Polygon):
            geometry = GeoJSONPolygon.from_shapely(shape)
        elif isinstance(shape, shapely.MultiPolygon):
            geometry = GeoJSONMultiPolygon.from_shapely(shape)
        else:
            raise TypeError(f"Unsupported geometry type: {type(shape)}")
        return cls(geometry=geometry, properties=properties)


def calculate_pcl_coverage(
    sensors: list[Sensor],
    grid: LatLonHeightGrid,
    rcs: float,
    snr_threshold: float = theia.config.SNR_THRESHOLD_PCL,
    doppler_threshold: float = theia.config.DOPPLER_SHIFT_THRESHOLD_PCL,
    delay_threshold: float = theia.config.DELAY_THRESHOLD_PCL,
) -> tuple[GeoJSONFeature, GeoJSONFeature]:
    """
    Calculate PCL coverage.

    Parameters
    ----------
    sensor: Sensor
        Sensor
    grid: LatLonHeightGrid
        Calculation grid
    rcs: float
        Radar cross section for which to calculate the coverage
    snr_threshold: float, default theia.config.SNR_THRESHOLD_PCL
        Minimum detectable threshold [dB]
    doppler_threshold: float, default theia.config.DOPPLER_SHIFT_THRESHOLD_PCL
        Minimum detectable Doppler shift [Hz]
    delay_threshold: float, default theia.config.DELAY_THRESHOLD_PCL
        Delay threshold for PCL [us].
        This is used to judge whether a given transmitter - target - receiver geometry
        is in the bistatic or the forward scattering regime.

    Returns
    -------
    track_init_coverage: GeoJSONFeature
        Region in which a track init can happen only using PCL
    track_update_coverage: GeoJSONFeature
        Region in which a track update can happen only using PCL
    """
    assert grid.altitude_values.shape[0] == 1

    detector = PclDetector(
        snr_threshold=snr_threshold,
        doppler_threshold=doppler_threshold,
        delay_threshold=delay_threshold,
    )

    track_init_mask, track_update_mask = pcl_track_init_update_masks(
        detector,
        sensors,
        grid,
        rcs,
    )

    polygons_init = mask_to_polygon(
        track_init_mask[:, :, 0],
        grid.latitude_values[0],
        grid.latitude_values[1] - grid.latitude_values[0],
        grid.longitude_values[0],
        grid.longitude_values[1] - grid.longitude_values[0],
    )
    import json

    # with open("polygons.json", "w") as file:
    #     json.dump(polygons_init, file)

    polygons_update = mask_to_polygon(
        track_update_mask[:, :, 0],
        grid.latitude_values[0],
        grid.latitude_values[1] - grid.latitude_values[0],
        grid.longitude_values[0],
        grid.longitude_values[1] - grid.longitude_values[0],
    )
    return (
        GeoJSONFeature(
            geometry=GeoJSONMultiPolygon.from_shapely(
                shapely.MultiPolygon(polygons_init)
            )
        ),
        GeoJSONFeature(
            geometry=GeoJSONMultiPolygon.from_shapely(
                shapely.MultiPolygon(polygons_update)
            )
        ),
    )


track_init, track_update = calculate_pcl_coverage([sensors[0]], grid, 1.0)

In [ ]:
from pydantic import TypeAdapter
from theia.types import Sensor

TypeAdapter(list[Sensor]).dump_json(sensors)

In [ ]:
sensors[0].model_dump_json()

In [ ]:
import cProfile

from theia.detection.pcl import pcl_track_init_update_masks

detector = PclDetector()

profiler = cProfile.Profile()
profiler.enable()

track_init_mask, track_update_mask = pcl_track_init_update_masks(
    detector,
    sensors,
    grid,
    RCS,
)

profiler.disable()
profiler.dump_stats("profile__pcl_coverage.prof")

In [ ]:
track_init_mask.shape

In [ ]:
grid.model_dump_json()

In [ ]:
import folium

from theia.mapping import RadarMap
from theia.util import mask_to_polygon


map = RadarMap(
    sensors={f"Sensor {sensor.id}": sensor for sensor in sensors},
    trajectories={"Target": trajectory},
).to_map()
map.location = (sensors[0].receiver.lat, sensors[0].receiver.lon)

polygons = mask_to_polygon(
    track_init_mask[:, :, 0],
    grid.latitude_values[0],
    grid.latitude_values[1] - grid.latitude_values[0],
    grid.longitude_values[0],
    grid.longitude_values[1] - grid.longitude_values[0],
)
for polygon in polygons:
    folium.GeoJson(polygon, fillColor="red", color="red").add_to(map)

polygons = mask_to_polygon(
    track_update_mask[:, :, 0],
    grid.latitude_values[0],
    grid.latitude_values[1] - grid.latitude_values[0],
    grid.longitude_values[0],
    grid.longitude_values[1] - grid.longitude_values[0],
)
for polygon in polygons:
    folium.GeoJson(polygon, fillColor="blue", color="blue").add_to(map)
map

In [ ]:
import numpy as np

from theia.detection.pcl import PclDetector
from theia.export_paraview import ParaviewExporter, PointOfInterest


pois: list[PointOfInterest] = []
poi_id = 0
for sensor in pcl_sensors:
    pois.append(
        PointOfInterest(
            id=poi_id,
            label=f"Rx {sensor.receiver.id}",
            type="Rx",
            lat=sensor.receiver.lat,
            lon=sensor.receiver.lon,
            alt=sensor.receiver.alt,
        )
    )
    poi_id += 1
    pois.append(
        PointOfInterest(
            id=poi_id,
            label=f"Tx {sensor.transmitter.id}",
            type="Tx",
            lat=sensor.transmitter.lat,
            lon=sensor.transmitter.lon,
            alt=sensor.transmitter.alt,
        )
    )
    poi_id += 1

exporter = ParaviewExporter(
    lat_min,
    lat_max,
    lat_res,
    lon_min,
    lon_max,
    lon_res,
)

sensor = pcl_sensors[0]

detector = PclDetector()

exporter.export_terrain(
    "test",
    # {
    #     "min detectable RCS": lambda *p: detector.minimum_detectable_rcs_vector(
    #         sensor.receiver, sensor.transmitter, np.array(p).reshape((1, 3))
    #     )
    # },
)
exporter.export_pois(pois, "pois.csv")